### Schema Validation
###### 1) Schema Order Validation
###### 2) Data Type Validation
###### 3) Column name validation
###### 4) Null value validation
###### 5) New Column Validation

### Schema Order Change

In [0]:
Drop table if exists deltacatalog.deltadb.invoices_sv;
Create or replace table deltacatalog.deltadb.invoices_sv
(
  customer_id int not null,
  invoice_no string,
  quantity int,
  payment_method string
)

In [0]:
insert into deltacatalog.deltadb.invoices_sv
select customer_id,invoice_no,quantity,payment_method from parquet.`abfss://labdata@stgdatabricksdelta.dfs.core.windows.net/invoices/invoices_1_100.parquet`
where customer_id <= 5

In [0]:
insert into deltacatalog.deltadb.invoices_sv
select quantity,invoice_no,customer_id,payment_method from parquet.`abfss://labdata@stgdatabricksdelta.dfs.core.windows.net/invoices/invoices_1_100.parquet`
where customer_id between 6 and 10

In [0]:
merge into deltacatalog.deltadb.invoices_sv tgt
using (select 100 as quantity, 'I12345' as invoice_no, 11 as customer_id, 'Credit Card' as payment_method) src
on tgt.customer_id = src.customer_id
when matched then update set tgt.payment_method = src.payment_method, tgt.quantity = src.quantity
when not matched then insert *

#### Data Type Validation

In [0]:
insert into deltacatalog.deltadb.invoices_sv
values ("ABC","I09876",678,"Cash")

In [0]:
insert into deltacatalog.deltadb.invoices_sv
values ("999999","I09876",678,"Cash")

In [0]:
Select * from deltacatalog.deltadb.invoices_sv

#### Column name validation

In [0]:
insert into deltacatalog.deltadb.invoices_sv
select 100 as Cust_id, '12890' as Invoice, 500 as Quan, 'Cash' as Payment

In [0]:
select * from deltacatalog.deltadb.invoices_sv

In [0]:
MERGE INTO deltacatalog.deltadb.invoices_sv tgt
USING (select customer_id,invoice_no,quantity,payment_method
from values (200,"I87536",2000,"Cash")
as T(customer_id,invoice_no,quantity,payment_method)) src
on tgt.customer_id = src.customer_id
when MATCHED then 
UPDATE 
SET 
tgt.payment_method = src.payment_method,
tgt.quantity = src.quantity,
tgt.invoice_no = src.invoice_no
when not matched then insert *

#### Null Validation

In [0]:
insert into deltacatalog.deltadb.invoices_sv values (Null,Null,Null,Null)

In [0]:
insert into deltacatalog.deltadb.invoices_sv values (12345,Null,Null,Null)

#### New Column Validation

In [0]:
insert into deltacatalog.deltadb.invoices_sv
select customer_id,invoice_no,quantity,payment_method,100 as cust_type
from values (5000,"I00000",50,"Credit Card")
as T(customer_id,invoice_no,quantity,payment_method)  

In [0]:
Merge into deltacatalog.deltadb.invoices_sv tgt
using (select customer_id,invoice_no,quantity,payment_method,"VIP" as cust_type
from values (321,"I898765",20,"Credit Card")
as T(customer_id,invoice_no,quantity,payment_method)
) as src
on tgt.customer_id = src.customer_id
when matched then 
update set tgt.payment_method = src.payment_method,
tgt.quantity = src.quantity,
tgt.invoice_no = src.invoice_no
when not matched then insert *

In [0]:
select * from deltacatalog.deltadb.invoices_sv